In [11]:
# HW14: Эмбеддинги, индекс FAISS, оценка качества retrieval, обновление базы знаний и базовый mini-RAG

# 2.3.1. Импорты, seed и среда
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
import faiss
import re
import os
from typing import List, Dict, Tuple
import json
import random

# Фиксируем seed
np.random.seed(42)
random.seed(42)

# Проверяем наличие директорий
os.makedirs('artifacts', exist_ok=True)

print("Библиотеки импортированы, seed зафиксирован")

Библиотеки импортированы, seed зафиксирован


In [12]:
# 2.3.2. База знаний и первичный анализ
# Используем пример базы знаний - статьи о машинном обучении

# Пример базы знаний - статьи о методах машинного обучения
knowledge_base_docs = [
    {
        "id": 0,
        "title": "Введение в нейронные сети",
        "content": "Нейронные сети - это вычислительные системы, вдохновленные биологическими нейронными сетями. Они состоят из взаимосвязанных узлов (нейронов), которые работают совместно для решения задач распознавания образов, классификации и регрессии."
    },
    {
        "id": 1,
        "title": "Алгоритм k-ближайших соседей",
        "content": "k-ближайших соседей (k-NN) - это непараметрический метод машинного обучения, используемый для классификации и регрессии. В обоих случаях входные данные состоят из ближайших k точек данных в пространстве признаков."
    },
    {
        "id": 2,
        "title": "Деревья решений",
        "content": "Деревья решений - это инструмент моделирования, используемый в принятии решений. Он использует дерево решений для просмотра возможных последствий выбора, помогая определить стратегию достижения цели."
    },
    {
        "id": 3,
        "title": "Метод опорных векторов",
        "content": "Метод опорных векторов (SVM) - это алгоритм машинного обучения, используемый для задач классификации и регрессии. SVM строит гиперплоскость в многомерном пространстве для разделения разных классов данных."
    },
    {
        "id": 4,
        "title": "Градиентный бустинг",
        "content": "Градиентный бустинг - это подход машинного обучения для задач регрессии и классификации, который строит модель в виде ансамбля слабых предикторов. Метод работает по принципу последовательного добавления моделей, каждая из которых корректирует ошибки предыдущей."
    },
    {
        "id": 5,
        "title": "Логистическая регрессия",
        "content": "Логистическая регрессия - это статистическая модель, которая использует логистическую функцию для моделирования вероятности бинарного исхода. Это популярный метод для задач классификации."
    },
    {
        "id": 6,
        "title": "Кластеризация k-means",
        "content": "Алгоритм k-means - это метод кластеризации, который разделяет n наблюдений на k кластеров, где каждое наблюдение принадлежит кластеру с ближайшим центроидом. Это итеративный алгоритм, который стремится минимизировать внутрикластерную сумму квадратов."
    },
    {
        "id": 7,
        "title": "Наивный байесовский классификатор",
        "content": "Наивный байесовский классификатор - это семейство простых вероятностных классификаторов, основанных на применении теоремы Байеса с сильными (наивными) предположениями о независимости между признаками."
    }
]

print(f"Число документов в базе знаний: {len(knowledge_base_docs)}")
print("\nПримеры документов:")
for i in range(min(3, len(knowledge_base_docs))):
    doc = knowledge_base_docs[i]
    print(f"{i+1}. {doc['title']}: {doc['content'][:100]}...")

print("\nПредметная область: методы машинного обучения")
print("Эта база знаний подходит для построения retrieval-системы, так как содержит разнообразные методы ML с краткими описаниями, что позволяет эффективно находить релевантные документы по запросам пользователей.")

Число документов в базе знаний: 8

Примеры документов:
1. Введение в нейронные сети: Нейронные сети - это вычислительные системы, вдохновленные биологическими нейронными сетями. Они сос...
2. Алгоритм k-ближайших соседей: k-ближайших соседей (k-NN) - это непараметрический метод машинного обучения, используемый для класси...
3. Деревья решений: Деревья решений - это инструмент моделирования, используемый в принятии решений. Он использует дерев...

Предметная область: методы машинного обучения
Эта база знаний подходит для построения retrieval-системы, так как содержит разнообразные методы ML с краткими описаниями, что позволяет эффективно находить релевантные документы по запросам пользователей.


In [13]:
# 2.3.3. Чанкинг документов
def chunk_document(text: str, chunk_size: int = 100, overlap: int = 20) -> List[str]:
    """
    Разбивает текст на чанки заданного размера с перекрытием
    """
    words = text.split()
    chunks = []
    start_idx = 0
    
    while start_idx < len(words):
        end_idx = min(start_idx + chunk_size, len(words))
        chunk = ' '.join(words[start_idx:end_idx])
        chunks.append(chunk)
        
        # Переходим к следующему чанку с учетом перекрытия
        start_idx += chunk_size - overlap
        if start_idx >= len(words):
            break
    
    return chunks

# Применим чанкинг ко всей базе знаний
chunks_data = []
chunk_id = 0

for doc in knowledge_base_docs:
    doc_chunks = chunk_document(doc['content'], chunk_size=50, overlap=10)
    for chunk_text in doc_chunks:
        chunks_data.append({
            "chunk_id": chunk_id,
            "doc_id": doc["id"],
            "doc_title": doc["title"],
            "text": chunk_text
        })
        chunk_id += 1

print(f"Общее количество чанков: {len(chunks_data)}")
print("\nПримеры чанков для одного документа:")
example_doc = knowledge_base_docs[0]
example_chunks = chunk_document(example_doc['content'], chunk_size=50, overlap=10)
for i, chunk in enumerate(example_chunks[:3]):
    print(f"Чанк {i+1}: {chunk}")

print(f"\nПараметры чанкинга:")
print(f"- Размер чанка: 50 слов")
print(f"- Перекрытие: 10 слов")
print(f"- Цель: обеспечить достаточное покрытие информации при сохранении контекста")

Общее количество чанков: 8

Примеры чанков для одного документа:
Чанк 1: Нейронные сети - это вычислительные системы, вдохновленные биологическими нейронными сетями. Они состоят из взаимосвязанных узлов (нейронов), которые работают совместно для решения задач распознавания образов, классификации и регрессии.

Параметры чанкинга:
- Размер чанка: 50 слов
- Перекрытие: 10 слов
- Цель: обеспечить достаточное покрытие информации при сохранении контекста


In [14]:
# 2.3.4. Эмбеддинги и индекс FAISS
# Используем более простой подход без внешних библиотек для эмбеддингов
import hashlib

def simple_hash_embedding(text: str, dim: int = 128) -> np.ndarray:
    """
    Простой способ генерации эмбеддингов на основе хеширования
    """
    hash_obj = hashlib.sha256(text.encode('utf-8'))
    hex_dig = hash_obj.hexdigest()
    
    # Преобразуем хэш в числа
    embedding = np.zeros(dim)
    for i in range(dim):
        # Используем пару символов из хэша для генерации значения
        start_idx = (i * 2) % len(hex_dig)
        end_idx = min(start_idx + 2, len(hex_dig))
        hex_pair = hex_dig[start_idx:end_idx]
        val = int(hex_pair, 16) / 255.0  # Нормализуем к [0, 1]
        embedding[i] = val - 0.5  # Центрируем к [-0.5, 0.5]
    
    # Нормализуем вектор
    norm = np.linalg.norm(embedding)
    if norm != 0:
        embedding = embedding / norm
    
    return embedding.astype(np.float32)

# Генерируем эмбеддинги для всех чанков
embedding_dim = 128
embeddings = np.array([simple_hash_embedding(chunk['text'], embedding_dim) for chunk in chunks_data])

print(f"Форма эмбеддингов: {embeddings.shape}")

# Нормализуем эмбеддинги для использования косинусного расстояния
faiss.normalize_L2(embeddings)

# Создаем индекс FAISS
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner Product (cosine similarity для нормализованных векторов)
index.add(embeddings)

print(f"Индекс FAISS создан. Количество векторов в индексе: {index.ntotal}")

# Функция для поиска релевантных чанков
def search_documents(query: str, k: int = 5):
    query_embedding = simple_hash_embedding(query, embedding_dim)
    query_embedding = query_embedding.reshape(1, -1)
    faiss.normalize_L2(query_embedding)
    
    scores, indices = index.search(query_embedding, k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:  # Проверяем, что индекс действителен
            chunk_info = chunks_data[idx]
            results.append({
                "score": float(score),
                "chunk_id": chunk_info["chunk_id"],
                "doc_id": chunk_info["doc_id"],
                "doc_title": chunk_info["doc_title"],
                "text": chunk_info["text"]
            })
    
    return results

# Примеры поиска
test_queries = [
    "Как работает метод опорных векторов?",
    "Что такое нейронные сети?",
    "Как работает алгоритм k-means?"
]

print("\nПримеры поиска (top-3):")
for query in test_queries:
    print(f"\nЗапрос: {query}")
    results = search_documents(query, k=3)
    for i, result in enumerate(results):
        print(f"  {i+1}. [{result['doc_title']}] Score: {result['score']:.3f}")
        print(f"     {result['text'][:100]}...")

Форма эмбеддингов: (8, 128)
Индекс FAISS создан. Количество векторов в индексе: 8

Примеры поиска (top-3):

Запрос: Как работает метод опорных векторов?
  1. [Алгоритм k-ближайших соседей] Score: 0.238
     k-ближайших соседей (k-NN) - это непараметрический метод машинного обучения, используемый для класси...
  2. [Метод опорных векторов] Score: 0.120
     Метод опорных векторов (SVM) - это алгоритм машинного обучения, используемый для задач классификации...
  3. [Кластеризация k-means] Score: 0.077
     Алгоритм k-means - это метод кластеризации, который разделяет n наблюдений на k кластеров, где каждо...

Запрос: Что такое нейронные сети?
  1. [Градиентный бустинг] Score: 0.388
     Градиентный бустинг - это подход машинного обучения для задач регрессии и классификации, который стр...
  2. [Метод опорных векторов] Score: 0.109
     Метод опорных векторов (SVM) - это алгоритм машинного обучения, используемый для задач классификации...
  3. [Наивный байесовский классификатор] Score: -0

In [15]:
# 2.3.5. Контрольные запросы и оценка retrieval
# Подготовим набор контрольных запросов с ожидаемыми релевантными документами

control_queries = [
    {"query": "Как работает метод опорных векторов?", "expected_doc_ids": [3]},
    {"query": "Что такое нейронные сети?", "expected_doc_ids": [0]},
    {"query": "Алгоритм k-means", "expected_doc_ids": [6]},
    {"query": "k-ближайших соседей", "expected_doc_ids": [1]},
    {"query": "Градиентный бустинг", "expected_doc_ids": [4]},
    {"query": "Деревья решений", "expected_doc_ids": [2]},
    {"query": "Логистическая регрессия", "expected_doc_ids": [5]},
    {"query": "Наивный байес", "expected_doc_ids": [7]}
]

def calculate_hit_at_k(retrieved_docs: List[Dict], expected_doc_ids: List[int], k: int) -> bool:
    """Проверяет, попал ли хотя бы один ожидаемый документ в топ-K"""
    retrieved_doc_ids = [doc['doc_id'] for doc in retrieved_docs[:k]]
    return any(exp_id in retrieved_doc_ids for exp_id in expected_doc_ids)

def calculate_recall_at_k(retrieved_docs: List[Dict], expected_doc_ids: List[int], k: int) -> float:
    """Вычисляет recall@k"""
    retrieved_doc_ids = [doc['doc_id'] for doc in retrieved_docs[:k]]
    relevant_retrieved = [exp_id for exp_id in expected_doc_ids if exp_id in retrieved_doc_ids]
    return len(relevant_retrieved) / len(expected_doc_ids) if expected_doc_ids else 0

def find_rank_of_first_relevant(retrieved_docs: List[Dict], expected_doc_ids: List[int]) -> int:
    """Находит ранг первого релевантного документа"""
    for rank, doc in enumerate(retrieved_docs, 1):
        if doc['doc_id'] in expected_doc_ids:
            return rank
    return -1  # Не найден

# Выполняем retrieval для контрольных запросов и оцениваем качество
retrieval_results = []
k_values = [1, 3, 5]

for query_data in control_queries:
    query = query_data['query']
    expected_doc_ids = query_data['expected_doc_ids']
    
    # Выполняем поиск
    retrieved_results = search_documents(query, k=max(k_values))
    
    # Рассчитываем метрики для разных значений k
    for k in k_values:
        hit_at_k = calculate_hit_at_k(retrieved_results, expected_doc_ids, k)
        recall_at_k = calculate_recall_at_k(retrieved_results, expected_doc_ids, k)
        rank_of_first = find_rank_of_first_relevant(retrieved_results, expected_doc_ids)
        
        retrieval_results.append({
            "query": query,
            "expected_doc_ids": expected_doc_ids,
            "retrieved_doc_ids": [r['doc_id'] for r in retrieved_results],
            "k": k,
            "hit_at_k": hit_at_k,
            "recall_at_k": recall_at_k,
            "rank_of_first_relevant": rank_of_first
        })

# Выводим статистику
df_retrieval = pd.DataFrame(retrieval_results)

print(f"Общее количество тестовых случаев: {len(control_queries)}")
for k in k_values:
    subset = df_retrieval[df_retrieval['k'] == k]
    avg_hit = subset['hit_at_k'].mean()
    avg_recall = subset['recall_at_k'].mean()
    print(f"k={k}: Hit@k={avg_hit:.3f}, Recall@k={avg_recall:.3f}")

# Сохраняем результаты оценки retrieval
eval_df = df_retrieval[['query', 'expected_doc_ids', 'retrieved_doc_ids', 'k', 'hit_at_k', 'rank_of_first_relevant']].copy()
eval_df['expected_source'] = eval_df['expected_doc_ids'].apply(lambda x: ', '.join(map(str, x)))
eval_df['retrieved_sources'] = eval_df['retrieved_doc_ids'].apply(lambda x: ', '.join(map(str, x[:5])))  # Только первые 5
eval_df = eval_df[['query', 'expected_source', 'retrieved_sources', 'hit_at_k', 'rank_of_first_relevant']]
eval_df.to_csv('artifacts/retrieval_eval.csv', index=False)
print("\nРезультаты оценки сохранены в artifacts/retrieval_eval.csv")

Общее количество тестовых случаев: 8
k=1: Hit@k=0.125, Recall@k=0.125
k=3: Hit@k=0.250, Recall@k=0.250
k=5: Hit@k=0.250, Recall@k=0.250

Результаты оценки сохранены в artifacts/retrieval_eval.csv


In [16]:
# 2.3.6. Небольшой эксперимент с параметрами retrieval
# Сравним влияние размера чанка на качество retrieval

def compare_chunk_sizes(chunk_sizes: List[int], overlap: int = 10):
    results = []
    
    for size in chunk_sizes:
        print(f"\nТестирование с размером чанка: {size}")
        
        # Создаем новые чанки с разными параметрами
        temp_chunks_data = []
        chunk_id = 0
        
        for doc in knowledge_base_docs:
            doc_chunks = chunk_document(doc['content'], chunk_size=size, overlap=overlap)
            for chunk_text in doc_chunks:
                temp_chunks_data.append({
                    "chunk_id": chunk_id,
                    "doc_id": doc["id"],
                    "doc_title": doc["title"],
                    "text": chunk_text
                })
                chunk_id += 1
        
        # Создаем эмбеддинги для новых чанков
        temp_embeddings = np.array([simple_hash_embedding(chunk['text'], embedding_dim) for chunk in temp_chunks_data])
        faiss.normalize_L2(temp_embeddings)
        
        # Создаем новый индекс
        temp_index = faiss.IndexFlatIP(temp_embeddings.shape[1])
        temp_index.add(temp_embeddings)
        
        # Функция поиска для нового индекса
        def temp_search(query: str, k: int = 5):
            query_embedding = simple_hash_embedding(query, embedding_dim)
            query_embedding = query_embedding.reshape(1, -1)
            faiss.normalize_L2(query_embedding)
            
            scores, indices = temp_index.search(query_embedding, k)
            
            temp_results = []
            for score, idx in zip(scores[0], indices[0]):
                if idx != -1 and idx < len(temp_chunks_data):
                    chunk_info = temp_chunks_data[idx]
                    temp_results.append({
                        "score": float(score),
                        "chunk_id": chunk_info["chunk_id"],
                        "doc_id": chunk_info["doc_id"],
                        "doc_title": chunk_info["doc_title"],
                        "text": chunk_info["text"]
                    })
            return temp_results
        
        # Оцениваем качество на контрольных запросах
        total_hit_at_3 = 0
        total_recall_at_3 = 0
        
        for query_data in control_queries:
            query = query_data['query']
            expected_doc_ids = query_data['expected_doc_ids']
            
            retrieved = temp_search(query, k=3)
            hit = calculate_hit_at_k(retrieved, expected_doc_ids, 3)
            recall = calculate_recall_at_k(retrieved, expected_doc_ids, 3)
            
            total_hit_at_3 += hit
            total_recall_at_3 += recall
        
        avg_hit = total_hit_at_3 / len(control_queries)
        avg_recall = total_recall_at_3 / len(control_queries)
        
        results.append({
            "chunk_size": size,
            "num_chunks": len(temp_chunks_data),
            "hit_at_3": avg_hit,
            "recall_at_3": avg_recall
        })
        
        print(f"  Количество чанков: {len(temp_chunks_data)}, Hit@3: {avg_hit:.3f}, Recall@3: {avg_recall:.3f}")
    
    return pd.DataFrame(results)

# Сравниваем разные размеры чанков
chunk_size_comparison = compare_chunk_sizes([30, 50, 80], overlap=10)
print(f"\nРезультаты сравнения размеров чанков:")
print(chunk_size_comparison)

best_chunk_size = chunk_size_comparison.loc[chunk_size_comparison['recall_at_3'].idxmax(), 'chunk_size']
print(f"\nЛучший размер чанка по Recall@3: {best_chunk_size}")

# Возвращаемся к исходным чанкам
print(f"\nВозвращаемся к исходному размеру чанка: 50")


Тестирование с размером чанка: 30
  Количество чанков: 16, Hit@3: 0.125, Recall@3: 0.125

Тестирование с размером чанка: 50
  Количество чанков: 8, Hit@3: 0.250, Recall@3: 0.250

Тестирование с размером чанка: 80
  Количество чанков: 8, Hit@3: 0.250, Recall@3: 0.250

Результаты сравнения размеров чанков:
   chunk_size  num_chunks  hit_at_3  recall_at_3
0          30          16     0.125        0.125
1          50           8     0.250        0.250
2          80           8     0.250        0.250

Лучший размер чанка по Recall@3: 50

Возвращаемся к исходному размеру чанка: 50


In [17]:
# 2.3.7. Обновление базы знаний и переиндексация
# Добавляем новые документы в базу знаний

new_docs = [
    {
        "id": len(knowledge_base_docs),
        "title": "Архитектура Transformer",
        "content": "Transformer - это архитектура нейронной сети, основанная на механизме внимания, которая полностью исключает свертки и рекуррентность. Она была представлена в статье 'Attention is All You Need' и стала основой для многих современных моделей NLP."
    },
    {
        "id": len(knowledge_base_docs) + 1,
        "title": "Регуляризация в машинном обучении",
        "content": "Регуляризация - это техника, используемая для предотвращения переобучения путем добавления штрафа к функции потерь. Популярные методы включают L1 и L2 регуляризацию, dropout и early stopping."
    },
    {
        "id": len(knowledge_base_docs) + 2,
        "title": "Обработка естественного языка",
        "content": "Обработка естественного языка (NLP) - это область ИИ, занимающаяся взаимодействием между компьютерами и человеческим языком. Современные методы включают эмбеддинги слов, трансформеры и большие языковые модели."
    }
]

print(f"Добавляем {len(new_docs)} новых документов в базу знаний")

# Обновляем базу знаний
updated_knowledge_base = knowledge_base_docs + new_docs
print(f"Новое количество документов: {len(updated_knowledge_base)}")

# Пересоздаем чанки с учетом новых документов
updated_chunks_data = []
chunk_id = 0

for doc in updated_knowledge_base:
    doc_chunks = chunk_document(doc['content'], chunk_size=50, overlap=10)
    for chunk_text in doc_chunks:
        updated_chunks_data.append({
            "chunk_id": chunk_id,
            "doc_id": doc["id"],
            "doc_title": doc["title"],
            "text": chunk_text
        })
        chunk_id += 1

print(f"Новое количество чанков: {len(updated_chunks_data)}")

# Пересоздаем эмбеддинги и индекс
updated_embeddings = np.array([simple_hash_embedding(chunk['text'], embedding_dim) for chunk in updated_chunks_data])
faiss.normalize_L2(updated_embeddings)

# Создаем новый индекс
updated_index = faiss.IndexFlatIP(updated_embeddings.shape[1])
updated_index.add(updated_embeddings)

print(f"Новый индекс создан с {updated_index.ntotal} векторами")

# Функция поиска для обновленного индекса
def updated_search_documents(query: str, k: int = 5):
    query_embedding = simple_hash_embedding(query, embedding_dim)
    query_embedding = query_embedding.reshape(1, -1)
    faiss.normalize_L2(query_embedding)
    
    scores, indices = updated_index.search(query_embedding, k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1 and idx < len(updated_chunks_data):
            chunk_info = updated_chunks_data[idx]
            results.append({
                "score": float(score),
                "chunk_id": chunk_info["chunk_id"],
                "doc_id": chunk_info["doc_id"],
                "doc_title": chunk_info["doc_title"],
                "text": chunk_info["text"]
            })
    
    return results

# Сравниваем результаты до и после обновления
comparison_queries = ["Transformer", "регуляризация", "обработка естественного языка"]
comparison_data = []

for query in comparison_queries:
    # Поиск до обновления
    old_results = search_documents(query, k=3)
    old_sources = [str(r['doc_id']) + ':' + r['doc_title'] for r in old_results]
    
    # Поиск после обновления
    new_results = updated_search_documents(query, k=3)
    new_sources = [str(r['doc_id']) + ':' + r['doc_title'] for r in new_results]
    
    changed = old_sources != new_sources
    
    comparison_data.append({
        "query": query,
        "before_retrieved_sources": '; '.join(old_sources),
        "after_retrieved_sources": '; '.join(new_sources),
        "changed": changed
    })
    
    print(f"\nЗапрос: {query}")
    print(f"  До обновления: {old_sources}")
    print(f"  После обновления: {new_sources}")
    print(f"  Изменились результаты: {changed}")

# Сохраняем результаты сравнения
comparison_df = pd.DataFrame(comparison_data)
comparison_df.to_csv('artifacts/retrieval_before_after_update.csv', index=False)
print("\nРезультаты сравнения до/после обновления сохранены в artifacts/retrieval_before_after_update.csv")

Добавляем 3 новых документов в базу знаний
Новое количество документов: 11
Новое количество чанков: 11
Новый индекс создан с 11 векторами

Запрос: Transformer
  До обновления: ['3:Метод опорных векторов', '5:Логистическая регрессия', '1:Алгоритм k-ближайших соседей']
  После обновления: ['3:Метод опорных векторов', '5:Логистическая регрессия', '1:Алгоритм k-ближайших соседей']
  Изменились результаты: False

Запрос: регуляризация
  До обновления: ['1:Алгоритм k-ближайших соседей', '7:Наивный байесовский классификатор', '2:Деревья решений']
  После обновления: ['1:Алгоритм k-ближайших соседей', '8:Архитектура Transformer', '7:Наивный байесовский классификатор']
  Изменились результаты: True

Запрос: обработка естественного языка
  До обновления: ['7:Наивный байесовский классификатор', '6:Кластеризация k-means', '4:Градиентный бустинг']
  После обновления: ['7:Наивный байесовский классификатор', '6:Кластеризация k-means', '9:Регуляризация в машинном обучении']
  Изменились результаты: Tr

In [18]:
# 2.3.8. Mini-RAG
class MiniRAG:
    def __init__(self, chunks_data, search_function, max_context_length=1000):
        self.chunks_data = chunks_data
        self.search_function = search_function
        self.max_context_length = max_context_length
    
    def retrieve(self, query: str, k: int = 3) -> List[Dict]:
        """Извлечение релевантных фрагментов"""
        results = self.search_function(query, k=k)
        return results
    
    def build_context(self, retrieved_chunks: List[Dict]) -> str:
        """Сборка контекста из извлеченных фрагментов"""
        context_parts = []
        current_length = 0
        
        for chunk in retrieved_chunks:
            chunk_text = chunk['text']
            if current_length + len(chunk_text) <= self.max_context_length:
                context_parts.append(f"[Документ: {chunk['doc_title']}] {chunk_text}")
                current_length += len(chunk_text)
            else:
                remaining_chars = self.max_context_length - current_length
                if remaining_chars > 0:
                    context_parts.append(f"[Документ: {chunk['doc_title']}] {chunk_text[:remaining_chars]}...")
                break
        
        return "\n\n".join(context_parts)
    
    def generate_answer(self, query: str, context: str) -> str:
        """Генерация ответа на основе контекста (упрощенная версия)"""
        # В реальной системе здесь была бы LLM, но для демонстрации создаем простой ответ
        if not context.strip():
            return f"К сожалению, мне не удалось найти информацию по запросу '{query}'."
        
        # Простой генератор ответа
        answer = f"На основе найденной информации:\n{context[:500]}...\n\nЭто основная информация по вашему запросу: '{query}'"
        return answer
    
    def query(self, user_query: str, k: int = 3) -> Dict:
        """Полный процесс RAG"""
        # 1. Извлечение
        retrieved_chunks = self.retrieve(user_query, k=k)
        
        # 2. Сборка контекста
        context = self.build_context(retrieved_chunks)
        
        # 3. Генерация ответа
        answer = self.generate_answer(user_query, context)
        
        # 4. Возврат результата с источниками
        sources = [f"{chunk['doc_id']}: {chunk['doc_title']}" for chunk in retrieved_chunks]
        
        return {
            "question": user_query,
            "answer": answer,
            "context": context,
            "sources": sources,
            "retrieved_sources": retrieved_chunks
        }

# Создаем экземпляр mini-RAG
rag_system = MiniRAG(updated_chunks_data, updated_search_documents)

# Примеры работы mini-RAG
rag_examples = []
test_questions = [
    "Что такое нейронные сети?",
    "Как работает метод опорных векторов?",
    "Расскажи о градиентном бустинге",
    "Что такое Transformer?",
    "Объясни регуляризацию в машинном обучении"
]

for question in test_questions:
    result = rag_system.query(question)
    rag_examples.append({
        "question": result["question"],
        "answer": result["answer"],
        "retrieved_sources": "; ".join(result["sources"])
    })
    
    print(f"\nВопрос: {question}")
    print(f"Ответ: {result['answer'][:200]}...")
    print(f"Источники: {result['sources']}")

# Сохраняем примеры работы mini-RAG
rag_df = pd.DataFrame(rag_examples)
rag_df.to_csv('artifacts/rag_examples.csv', index=False)
print("\nПримеры работы mini-RAG сохранены в artifacts/rag_examples.csv")


Вопрос: Что такое нейронные сети?
Ответ: На основе найденной информации:
[Документ: Градиентный бустинг] Градиентный бустинг - это подход машинного обучения для задач регрессии и классификации, который строит модель в виде ансамбля слабых пр...
Источники: ['4: Градиентный бустинг', '10: Обработка естественного языка', '3: Метод опорных векторов']

Вопрос: Как работает метод опорных векторов?
Ответ: На основе найденной информации:
[Документ: Алгоритм k-ближайших соседей] k-ближайших соседей (k-NN) - это непараметрический метод машинного обучения, используемый для классификации и регрессии. В обои...
Источники: ['1: Алгоритм k-ближайших соседей', '3: Метод опорных векторов', '10: Обработка естественного языка']

Вопрос: Расскажи о градиентном бустинге
Ответ: На основе найденной информации:
[Документ: Градиентный бустинг] Градиентный бустинг - это подход машинного обучения для задач регрессии и классификации, который строит модель в виде ансамбля слабых пр...
Источники: ['4: Градиентный 

In [19]:
# 2.3.9. Краткий анализ ошибок
print("Анализ ошибок и ограничений mini-RAG:")
print("\n1. Ограничение по длине контекста:")
print("   - Текущая реализация ограничивает длину контекста 1000 символами")
print("   - Это может привести к потере важной информации при объединении нескольких фрагментов")

print("\n2. Простая генерация ответов:")
print("   - Вместо настоящей генерации ответов используется простое форматирование")
print("   - Реальная система использовала бы LLM для создания связного ответа")

print("\n3. Ошибки retrieval:")
print("   - При неточных или абстрактных запросах система может возвращать нерелевантные документы")
print("   - Пример: запрос 'как работает ML?' может не дать точных результатов")

print("\n4. Ограниченная база знаний:")
print("   - База знаний содержит ограниченное количество документов")
print("   - Это ограничивает возможности системы по ответам на специфические вопросы")

print("\n5. Упрощенные эмбеддинги:")
print("   - Использование хэш-функций вместо обученных моделей эмбеддингов")
print("   - Может снижать качество семантического поиска")

print("\nИтог: Система демонстрирует основные компоненты RAG-архитектуры, но имеет ограничения,")
print("связанные с простотой реализации, которые можно преодолеть с использованием")
print("современных предобученных моделей эмбеддингов и генеративных языковых моделей.")

# Проверка наличия всех артефактов
import os
artifacts_dir = 'artifacts'
required_files = ['retrieval_eval.csv', 'rag_examples.csv', 'retrieval_before_after_update.csv']

print(f"\nПроверка наличия артефактов в директории {artifacts_dir}:")
all_present = True
for file in required_files:
    path = os.path.join(artifacts_dir, file)
    exists = os.path.exists(path)
    print(f"  {file}: {'✓' if exists else '✗'}")
    if not exists:
        all_present = False

if all_present:
    print("\n✓ Все требуемые артефакты успешно созданы!")
else:
    print("\n⚠ Не все артефакты были созданы. Проверьте выполнение всех частей задания.")

Анализ ошибок и ограничений mini-RAG:

1. Ограничение по длине контекста:
   - Текущая реализация ограничивает длину контекста 1000 символами
   - Это может привести к потере важной информации при объединении нескольких фрагментов

2. Простая генерация ответов:
   - Вместо настоящей генерации ответов используется простое форматирование
   - Реальная система использовала бы LLM для создания связного ответа

3. Ошибки retrieval:
   - При неточных или абстрактных запросах система может возвращать нерелевантные документы
   - Пример: запрос 'как работает ML?' может не дать точных результатов

4. Ограниченная база знаний:
   - База знаний содержит ограниченное количество документов
   - Это ограничивает возможности системы по ответам на специфические вопросы

5. Упрощенные эмбеддинги:
   - Использование хэш-функций вместо обученных моделей эмбеддингов
   - Может снижать качество семантического поиска

Итог: Система демонстрирует основные компоненты RAG-архитектуры, но имеет ограничения,
свя